# 🏧 Notebook 1: ATM — Class Design

An **Automated Teller Machine (ATM)** lets a bank customer do things like **check balance**, **withdraw cash**, **deposit cash**, and **transfer funds** — without a human teller.

This notebook is about **thinking before coding**. We'll look at a *bad* design first, see why it hurts, then refactor step-by-step into a cleaner one. Notebook 2 implements the final design end-to-end.

### Requirements (from the Grokking problem)
- User inserts a **card** and enters a **PIN** to authenticate.
- A card is linked to one **customer** who can have two account types: **Checking** and **Savings**.
- Once authenticated, the customer can:
  1. Check balance
  2. Deposit cash
  3. Withdraw cash (from checking)
  4. Transfer funds between accounts
- The ATM has real **hardware**: a card reader, a keypad, a screen, a cash dispenser, a deposit slot, a printer.
- The ATM holds a limited amount of cash and refuses withdrawals it cannot fulfill.
- Wrong PIN three times → the card is blocked and ejected.
- There is a **daily withdrawal limit** per account.


## 🛠️ Setup

```bash
cd 07-object-oriented-design/atm
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 1️⃣ The *bad* design — one giant "God class"

A very common first attempt: put **everything** inside one class called `ATM`. It holds accounts, PINs, balances, the cash drawer, and all the business rules.


In [1]:
# ❌ BAD: one class does everything. Hard to read, hard to test, hard to extend.
class BadATM:
    def __init__(self):
        # accounts stored as a dict — ATM knows internal bank data (wrong!)
        self.accounts = {"A-1": {"pin": "1234", "balance": 500.0}}
        self.cash_in_machine = 1000.0
        self.current_account_id = None

    def do_everything(self, action, *args):
        # one method with a giant if/elif tree — painful to extend
        if action == "login":
            aid, pin = args
            if self.accounts[aid]["pin"] != pin:
                return "bad pin"
            self.current_account_id = aid
            return "ok"
        elif action == "withdraw":
            amt = args[0]
            acc = self.accounts[self.current_account_id]
            if amt > acc["balance"] or amt > self.cash_in_machine:
                return "no"
            acc["balance"] -= amt
            self.cash_in_machine -= amt
            return "ok"
        elif action == "balance":
            return self.accounts[self.current_account_id]["balance"]
        # ...and on and on for deposit, transfer, eject, receipt, etc.

bad = BadATM()
print(bad.do_everything("login", "A-1", "1234"))
print(bad.do_everything("withdraw", 100))
print(bad.do_everything("balance"))


ok
ok
400.0


### Why is this bad?

- **Single Responsibility Principle** violated: one class knows about hardware, accounts, PINs, *and* transaction rules.
- **Open/Closed Principle** violated: to add a *transfer* feature you must edit `do_everything` again.
- **No state machine**: nothing stops `withdraw` before `login`. The `current_account_id` is a fragile hidden flag.
- **Hard to test**: you cannot test "dispenser empty" without also setting up accounts and PINs.
- **Hard to reuse**: the "bank" logic is tangled with the "ATM hardware" logic. A real ATM talks to a *remote* bank — here they're the same object.


## 2️⃣ A better split — separate responsibilities

Good OOD groups code by **who is responsible for what**. For an ATM the natural split is:

| Group | Classes | Responsibility |
|---|---|---|
| **Hardware** | `CardReader`, `Keypad`, `Screen`, `CashDispenser`, `DepositSlot`, `Printer` | Physical input/output. Each knows *one* device. |
| **Bank side** | `Bank`, `Customer`, `Account` (`CheckingAccount`, `SavingsAccount`), `Card` | Stores money and customer data. Lives on the bank's servers in reality. |
| **Transactions** | `Transaction` (base) → `BalanceInquiry`, `Withdraw`, `Deposit`, `Transfer` | Each transaction is its own class (**Command pattern**). Adding a new one = new class, no giant `if`. |
| **Controller** | `ATM` | Orchestrates a session: talks to hardware, asks the bank to authenticate, runs a `Transaction`. Holds a **state machine**. |

### ASCII class diagram

```
             ┌──────────┐  uses   ┌────────────┐  talks to  ┌────────┐
             │   ATM    │────────▶│  Hardware  │            │  Bank  │
             └──────────┘         │ components │            └────────┘
                  │               └────────────┘                 │
                  │ runs                                          │ owns
                  ▼                                               ▼
          ┌───────────────┐  abstract            ┌──────────────────┐
          │  Transaction  │◀─── Withdraw         │     Customer     │
          └───────────────┘      Deposit         └──────────────────┘
                                 Transfer                 │ 1
                                 BalanceInquiry           │ *
                                                          ▼
                                                  ┌──────────────┐
                                                  │   Account    │◀─ Checking
                                                  └──────────────┘   Savings
```

### State machine of one session

```
IDLE ──insert_card──▶ CARD_INSERTED ──pin_ok──▶ AUTHENTICATED ──choose──▶ TRANSACTING
  ▲                       │ 3 wrong pins             │                      │
  │                       ▼                          │                      │
  └────────────────── card blocked ◀─────────────────┴───────── eject ◀─────┘
```


## 3️⃣ Tiny runnable skeleton

Let's sanity-check that the separation *compiles and runs*. Just skeletons — Notebook 2 adds the real logic.


In [2]:
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from enum import Enum

# ---- Bank side ------------------------------------------------------------
class AccountType(Enum):
    CHECKING = "checking"
    SAVINGS  = "savings"

@dataclass
class Account:
    id: str
    kind: AccountType
    balance: float = 0.0

# ---- Card & Customer ------------------------------------------------------
@dataclass
class Customer:
    name: str
    accounts: dict  # {AccountType: Account}

@dataclass
class Card:
    number: str
    pin: str
    customer: Customer

# ---- Hardware (stubs) -----------------------------------------------------
class CashDispenser:
    def __init__(self, cash): self.cash = cash

class Screen:
    def show(self, msg): print("screen>", msg)

class Printer:
    def print_receipt(self, text): print("receipt>", text)

# ---- Transactions (Command pattern) --------------------------------------
class Transaction(ABC):
    @abstractmethod
    def execute(self, atm): ...

class BalanceInquiry(Transaction):
    def __init__(self, account): self.account = account
    def execute(self, atm):
        atm.screen.show(f"Balance: ${self.account.balance:.2f}")

# Quick check: the shape works.
alice = Customer("Alice", {AccountType.CHECKING: Account("A-1", AccountType.CHECKING, 500)})
card  = Card("CARD-1", "1234", alice)
BalanceInquiry(alice.accounts[AccountType.CHECKING]).execute(
    type("FakeATM", (), {"screen": Screen()})()
)


screen> Balance: $500.00


## 4️⃣ What the split buys you

- **Add `Transfer`?** Write a new `Transfer(Transaction)` class. No existing code changes.
- **Swap the dispenser** for a fake one in tests? Pass a different `CashDispenser`-like object. This is **dependency injection**.
- **Block a card after 3 bad PINs?** The `Bank` owns that rule, not the ATM.
- **Two ATMs talk to the same bank?** Both get a `Bank` reference; account state is shared correctly.

👉 Notebook 2 puts all of this together into a working mini-ATM with tests.
